In [1]:
"""
Revenue Impact Simulation - Global Sales Performance Analysis
================================================================
Extension to the original analysis modeling three forward-looking (2025)
scenarios using actual CAGR (compound annual growth rate) calculated from
the cleaned 2022-2024 dataset. No figures in this script are assumed or
invented; all inputs are derived directly from Global_Sales_Performance_Cleaned.csv.

Scenarios modeled:
  1. Baseline      - each region/category continues its own 2022-2024 CAGR into 2025
  2. LatAm Stabilization - Latin America's decline is halted at 2024 level
  3. ME-led Recovery     - all regions match Middle East's CAGR (the strongest performer)
  4. Category Diversification - 10% of total revenue mix is shifted from the
     overconcentrated, declining Raw Materials category into the three
     categories with positive 2022-2024 CAGR (Apparel & Footwear, Automotive
     Parts, Consumer Goods), proportional to their existing relative size.
"""

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 12
BLUE = "#4C72B0"
GREEN = "#55A868"
ORANGE = "#DD8452"

df = pd.read_csv("data/Global_Sales_Performance_Cleaned.csv")

region_piv = df.pivot_table(index="Region", columns="Year", values="Total_Revenue", aggfunc="sum")
cat_piv = df.pivot_table(index="Product_Category", columns="Year", values="Total_Revenue", aggfunc="sum")


def cagr(v0, v1, periods):
    return (v1 / v0) ** (1 / periods) - 1


#  Regional CAGR (2022-2024)
region_cagr = {r: cagr(region_piv.loc[r, 2022], region_piv.loc[r, 2024], 2) for r in region_piv.index}
total_2024 = region_piv[2024].sum()

# Scenario 1: Baseline of each region continues its own trend into 2025
baseline_2025 = {r: region_piv.loc[r, 2024] * (1 + region_cagr[r]) for r in region_piv.index}
baseline_total_2025 = sum(baseline_2025.values())

# Scenario 2: Latin America stabilizes and decline halted (2024 level)
latam_stabilized_2025 = dict(baseline_2025)
latam_stabilized_2025["Latin America"] = region_piv.loc["Latin America", 2024]
latam_revenue_protected = sum(latam_stabilized_2025.values()) - baseline_total_2025

# Scenario 3: Middle East recovery, and all regions match Middle East's CAGR
me_cagr = region_cagr["Middle East"]
me_led_2025 = {r: region_piv.loc[r, 2024] * (1 + me_cagr) for r in region_piv.index}
me_led_total_2025 = sum(me_led_2025.values())
me_upside = me_led_total_2025 - baseline_total_2025

# Category CAGR (2022-2024)
cat_cagr = {c: cagr(cat_piv.loc[c, 2022], cat_piv.loc[c, 2024], 2) for c in cat_piv.index}
growing_cats = ["Apparel & Footwear", "Automotive Parts", "Consumer Goods"]
growing_total_2024 = sum(cat_piv.loc[c, 2024] for c in growing_cats)
shift_amount = total_2024 * 0.10

baseline_cat_2025 = {c: cat_piv.loc[c, 2024] * (1 + cat_cagr[c]) for c in cat_piv.index}
baseline_cat_total_2025 = sum(baseline_cat_2025.values())

diversified_2024 = dict(cat_piv[2024])
diversified_2024["Raw Materials"] -= shift_amount
for c in growing_cats:
    diversified_2024[c] += shift_amount * (cat_piv.loc[c, 2024] / growing_total_2024)
diversified_cat_2025 = {c: diversified_2024[c] * (1 + cat_cagr[c]) for c in cat_piv.index}
diversified_total_2025 = sum(diversified_cat_2025.values())
diversification_upside = diversified_total_2025 - baseline_cat_total_2025

# Printing summary
print("REGIONAL SCENARIOS (2025 projection)")
print(f"  Baseline total:            ${baseline_total_2025:,.0f}")
print(f"  LatAm stabilized total:    ${sum(latam_stabilized_2025.values()):,.0f}  (+${latam_revenue_protected:,.0f} protected)")
print(f"  ME-led recovery total:     ${me_led_total_2025:,.0f}  (+${me_upside:,.0f} upside)")
print()
print("CATEGORY DIVERSIFICATION SCENARIO (2025 projection)")
print(f"  Baseline total:            ${baseline_cat_total_2025:,.0f}")
print(f"  Diversified total:         ${diversified_total_2025:,.0f}  (+${diversification_upside:,.0f} upside)")

# Chart 1: Regional scenario comparison
fig, ax = plt.subplots(figsize=(8, 5.5))
labels = ["Baseline\n(trend continues)", "LatAm\nStabilized", "ME-led\nRecovery"]
values = [baseline_total_2025, sum(latam_stabilized_2025.values()), me_led_total_2025]
colors = [BLUE, GREEN, ORANGE]
bars = ax.bar(labels, values, color=colors)
ax.set_title("2025 Revenue Projection by Scenario")
ax.set_ylabel("Projected Total Revenue (USD)")
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 500000, f"${val/1e6:.1f}M", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("charts/chart08_revenue_impact_scenarios.png", dpi=150)
plt.close()

# Chart 2: Category diversification
fig, ax = plt.subplots(figsize=(7, 5.5))
labels2 = ["Baseline\n(no diversification)", "Diversified\n(10% shift from\nRaw Materials)"]
values2 = [baseline_cat_total_2025, diversified_total_2025]
bars2 = ax.bar(labels2, values2, color=[BLUE, GREEN])
ax.set_title("2025 Revenue Projection: Category Diversification")
ax.set_ylabel("Projected Total Revenue (USD)")
ax.set_ylim(min(values2) * 0.97, max(values2) * 1.02)
for bar, val in zip(bars2, values2):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 100000, f"${val/1e6:.2f}M", ha="center", fontsize=11)
plt.tight_layout()
plt.savefig("charts/chart09_category_diversification.png", dpi=150)
plt.close()

print("\nCharts saved: chart08_revenue_impact_scenarios.png, chart09_category_diversification.png")

FileNotFoundError: [Errno 2] No such file or directory: 'data/Global_Sales_Performance_Cleaned.csv'